# Quick SAMPIC waveform viewer

Handles empty and metadata-only events. Baseline and amplitude are estimated from the waveform: first 8 samples and peak-to-peak respectively. Calibrated voltages are displayed in mV.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run00266/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('SAMPIC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
UNIT = sampic_unit(tree)
print('Waveform display units:', UNIT)


In [ ]:
waveform_entries = find_waveform_entries(tree)
print(f'{tree.GetEntries()} entries; {len(waveform_entries)} contain waveform samples')
def plot_hit(entry=None, hit_index=0):
    if entry is None:
        entry = waveform_entries[0] if waveform_entries else None
    if entry is None:
        print('No waveforms available'); return
    if not 0 <= entry < tree.GetEntries(): raise IndexError(entry)
    tree.GetEntry(entry)
    if not present(tree, 'event') or not 0 <= hit_index < len(tree.sampic_event.hits):
        print('No such hit in this entry'); return
    hit = tree.sampic_event.hits[hit_index]
    wf = hit_waveform(hit)
    if not wf.size:
        print('Hit has no valid waveform samples'); return
    fig, ax = plt.subplots()
    ax.plot(wf)
    ax.axhline(hit_baseline(hit), color='green', linestyle='--', label='First-8-sample baseline estimate')
    ax.set(xlabel='Sample index', ylabel=UNIT,
           title=f'Entry {entry}, hit {hit_index}, FEB {hit.fe_board_index}, channel {hit.channel}')
    ax.legend(); plt.show()
plot_hit()


In [ ]:
amplitudes, channels, multiplicities = [], [], []
for idx in range(tree.GetEntries()):
    tree.GetEntry(idx)
    hits = tree.sampic_event.hits if present(tree, 'event') else []
    multiplicities.append(len(hits))
    for hit in hits:
        if len(hit.corrected_waveform): amplitudes.append(hit_amplitude(hit))
        channels.append((int(hit.fe_board_index), int(hit.channel)))
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].hist(amplitudes, bins=40); axes[0].set(xlabel=f'Peak-to-peak [{UNIT}]', ylabel='Hits')
counts = Counter(channels)
axes[1].bar([str(k) for k in sorted(counts)], [counts[k] for k in sorted(counts)])
axes[1].set(xlabel='(FEB, channel)', ylabel='Hits'); axes[1].tick_params(axis='x', rotation=90)
axes[2].hist(multiplicities, bins=40); axes[2].set(xlabel='Hits per MIDAS event', ylabel='Events')
plt.tight_layout(); plt.show()
print(f'Total hits: {len(channels):,}')
